# 02 · 流匹配与概率流 ODE  Flow Matching & Probability Flow ODE

> **能力标签**：GA3（扩散模型与流匹配 / Diffusion Models & Flow Matching）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释**流匹配**（Flow Matching）的核心思想：通过学习连续时间向量场（velocity field）将噪声分布 $p_0$ 变换为数据分布 $p_1$。
2. 描述**概率流 ODE**（probability-flow ODE）：$\frac{dx}{dt} = v_\theta(x, t)$，并与 DDPM 随机反向过程对比。
3. 推导 **OT 条件流匹配**（OT-CFM）：最优传输线性插值路径 $x_t = (1-t)x_0 + t\, x_1$ 及目标速度场 $v^* = x_1 - x_0$。
4. 实现 `ot_interpolate` 和 `velocity_target`，在合成数据上训练速度场网络并可视化轨迹。
5. 对比流匹配与 DDPM 的训练目标与采样流程差异。

> 对应能力：**GA3**

## 概念讲解 Concepts

### 1. 生成模型的"传输"视角 The Transport View of Generation

生成模型的本质是学习从简单分布（如标准高斯 $p_0 = \mathcal{N}(0,I)$）到数据分布（$p_1 = p_{\text{data}}$）的**变换**（transport）。

- **扩散模型**（DDPM）：通过随机微分方程（SDE）逐步去噪，采样是随机的（stochastic）。
- **流匹配**（Flow Matching）：学习确定性 ODE 向量场，采样是确定性的（deterministic）。

---

### 2. 连续正规化流 Continuous Normalizing Flow (CNF)

流匹配训练一个神经网络 $v_\theta(x, t)$ 使得 ODE

$$\frac{dx}{dt} = v_\theta(x, t), \quad x(0) \sim p_0$$

在 $t=1$ 时将 $p_0$ 变换为 $p_1$。推断时从 $z \sim p_0$ 出发，用 ODE 求解器积分到 $t=1$ 得到样本。

---

### 3. 条件流匹配目标 Conditional Flow Matching Objective

直接拟合边际速度场（marginal velocity field）很难。Lipman et al. (2022) 与 Albergo & Vanden-Eijnden (2022) 独立提出：用**条件速度场**（conditional velocity field）的期望来近似边际速度场，并证明两者的训练目标梯度相同。

**OT-CFM（Optimal Transport Conditional Flow Matching）**（Tong et al., 2023）用最优传输耦合 $(x_0, x_1) \sim \pi^*$ 构造条件路径，路径为**线性插值**（linear interpolation）：

$$x_t = (1-t)\, x_0 + t\, x_1, \quad t \in [0, 1]$$

目标速度场（velocity target）：

$$v^*(x_t, t \mid x_0, x_1) = \frac{dx_t}{dt} = x_1 - x_0$$

**训练目标**：

$$\mathcal{L}_{\text{CFM}} = \mathbb{E}_{t, (x_0, x_1), x_t}\!\left[\|v_\theta(x_t, t) - (x_1 - x_0)\|^2\right]$$

注意：目标速度 $x_1 - x_0$ 与时间 $t$ 无关（对线性路径而言），训练极为简洁。

---

### 4. 对比 DDPM 与流匹配 Comparison: DDPM vs Flow Matching

| 维度 | DDPM | OT-CFM |
|------|------|--------|
| 路径 | 随机 SDE 前向/反向 | 确定性 ODE，线性插值 |
| 训练目标 | $\|\epsilon - \epsilon_\theta(x_t, t)\|^2$ | $\|(x_1 - x_0) - v_\theta(x_t, t)\|^2$ |
| 采样 | 随机去噪（数百步） | ODE 积分（可少步） |
| 实现复杂度 | 需要噪声表 $\bar{\alpha}_t$ | 仅需线性插值 |
| 速度 | 较慢（大 T） | 更快（少步 ODE） |

---

### 5. 实现摘要 Implementation Summary

```python
# 线性插值 Linear interpolation (ot_interpolate)
x_t = (1 - t) * x0 + t * x1          # t: (B,1) or scalar

# 目标速度 Velocity target
v_target = x1 - x0                    # constant for linear paths

# 训练步 Training step
t = torch.rand(B, 1)
x_t = ot_interpolate(x0, x1, t)
v_pred = model(x_t, t)               # network predicts velocity
loss = F.mse_loss(v_pred, v_target)
```

## 示例 Worked Example

In [ ]:
# Setup
import math
import tempfile
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
DEVICE = torch.device('cpu')
TMPDIR = tempfile.mkdtemp()
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
print('Tmp dir:', TMPDIR)

In [ ]:
# ot_interpolate + velocity_target (mirrors w9-flow-matching)

def ot_interpolate(x0, x1, t):
    '''OT 条件流匹配的线性插值路径：x_t = (1-t) x0 + t x1。
    OT-CFM linear interpolation path.
    x0, x1: (B, D); t: (B, 1) or scalar in [0, 1].
    '''
    return (1.0 - t) * x0 + t * x1


def velocity_target(x0, x1):
    '''线性路径的目标速度场：v = dx_t/dt = x1 - x0（与 t 无关）。
    Velocity target for linear path: v = x1 - x0 (constant in t).
    x0, x1: (B, D). Returns (B, D).
    '''
    return x1 - x0


# --- 单元测试 Unit Tests ---
torch.manual_seed(0)
B, D = 8, 4
x0 = torch.randn(B, D)
x1 = torch.randn(B, D)

# t=0: x_t == x0
xt0 = ot_interpolate(x0, x1, t=0.0)
assert torch.allclose(xt0, x0, atol=1e-6), 'ot_interpolate(t=0) 应等于 x0'

# t=1: x_t == x1
xt1 = ot_interpolate(x0, x1, t=1.0)
assert torch.allclose(xt1, x1, atol=1e-6), 'ot_interpolate(t=1) 应等于 x1'

# t=0.5: 中点
xt_half = ot_interpolate(x0, x1, t=0.5)
expected_half = 0.5 * x0 + 0.5 * x1
assert torch.allclose(xt_half, expected_half, atol=1e-6), '中点错误'

# velocity_target shape
v_tgt = velocity_target(x0, x1)
assert v_tgt.shape == (B, D), '速度形状错误'
assert torch.allclose(v_tgt, x1 - x0, atol=1e-6), '速度目标错误'

print('\u2713 ot_interpolate 通过 passed')
print('\u2713 velocity_target 通过 passed')

In [ ]:
# 训练速度场网络 Train a velocity field network

torch.manual_seed(42)

# 合成数据 Synthetic data: source=Gaussian, target=mixture of 2 Gaussians
N = 512
x0_data = torch.randn(N, 2)                            # noise source
centers = torch.tensor([[2.0, 0.0], [-2.0, 0.0]])
idx = torch.randint(2, (N,))
x1_data = centers[idx] + 0.4 * torch.randn(N, 2)      # data target

# Velocity network: takes [x_t (2d), t (1d)] -> velocity (2d)
class VelocityNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2)
        )
    def forward(self, x, t):
        if t.dim() == 1:
            t = t.unsqueeze(1)
        return self.net(torch.cat([x, t], dim=1))

torch.manual_seed(42)
vnet = VelocityNet()
opt = torch.optim.Adam(vnet.parameters(), lr=1e-3)

losses = []
for step in range(1000):
    idx_b = torch.randint(N, (128,))
    x0_b = x0_data[idx_b]
    x1_b = x1_data[idx_b]

    t_b = torch.rand(128, 1)
    x_t_b = ot_interpolate(x0_b, x1_b, t_b)
    v_tgt_b = velocity_target(x0_b, x1_b)

    v_pred_b = vnet(x_t_b, t_b)
    loss = F.mse_loss(v_pred_b, v_tgt_b)

    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

print(f'最终损失 Final loss: {losses[-1]:.4f}  (初始 initial: {losses[0]:.4f})')
assert losses[-1] < losses[0], '损失应下降 Loss should decrease'
print('\u2713 速度场网络训练成功 Velocity net training passed')

In [ ]:
# 可视化：速度场 + 生成轨迹 Visualize velocity field + generation trajectories

def euler_integrate(vnet, x_start, n_steps=50):
    '''从 x_start 用欧拉法积分速度场至 t=1。
    Euler integration of velocity field from t=0 to t=1.
    '''
    x = x_start.clone()
    dt = 1.0 / n_steps
    for i in range(n_steps):
        t_val = torch.full((x.shape[0], 1), i * dt)
        with torch.no_grad():
            v = vnet(x, t_val)
        x = x + v * dt
    return x

torch.manual_seed(0)
n_test = 200
x0_test = torch.randn(n_test, 2)
x_gen = euler_integrate(vnet, x0_test, n_steps=50)

# Few trajectories
n_traj = 6
x0_traj = torch.randn(n_traj, 2)
trajectories = [x0_traj.clone()]
dt_traj = 1.0 / 20
x_cur = x0_traj.clone()
for i in range(20):
    t_v = torch.full((n_traj, 1), i * dt_traj)
    with torch.no_grad():
        v_step = vnet(x_cur, t_v)
    x_cur = x_cur + v_step * dt_traj
    trajectories.append(x_cur.clone())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Loss curve
axes[0].plot(losses)
axes[0].set_yscale('log')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('MSE Loss')
axes[0].set_title('OT-CFM 训练损失 Training Loss')

# Generated samples vs target
axes[1].scatter(x1_data[:, 0], x1_data[:, 1], s=6, alpha=0.3,
                color='coral', label='target $x_1$')
axes[1].scatter(x_gen[:, 0].detach(), x_gen[:, 1].detach(), s=6, alpha=0.5,
                color='steelblue', label='generated')
axes[1].set_title('OT-CFM 生成样本 Generated Samples')
axes[1].legend(fontsize=8)
axes[1].set_xlim(-4, 4); axes[1].set_ylim(-4, 4)

# Trajectories
colors_traj = plt.cm.viridis([i / n_traj for i in range(n_traj)])
for j in range(n_traj):
    traj_pts = torch.stack([s[j] for s in trajectories], dim=0).detach()
    axes[2].plot(traj_pts[:, 0], traj_pts[:, 1], '-o',
                 color=colors_traj[j], markersize=3, alpha=0.8, linewidth=1.2)
axes[2].scatter([x0_traj[j, 0].item() for j in range(n_traj)],
               [x0_traj[j, 1].item() for j in range(n_traj)],
               s=60, color='green', zorder=5, label='$x_0$')
axes[2].scatter([x_cur[j, 0].item() for j in range(n_traj)],
               [x_cur[j, 1].item() for j in range(n_traj)],
               s=60, color='red', zorder=5, marker='*', label='$x_1$')
axes[2].set_title('生成轨迹 Generation Trajectories')
axes[2].legend(fontsize=8)

plt.tight_layout()
fig_path = os.path.join(TMPDIR, 'flow_matching.png')
plt.savefig(fig_path, dpi=80); plt.close()
print(f'图像已保存 Saved: {fig_path}')

## 动手 Exercise

实现 `cfm_training_step`：对一批 $(x_0, x_1)$ 对，随机采样 $t \in [0,1]$，计算 $x_t$（`ot_interpolate`），让网络预测速度，返回 MSE 损失。注意使用 `(B, 1)` 形状的 $t$。


In [ ]:
# Exercise: CFM training step

def cfm_training_step(model, x0, x1):
    '''OT-CFM 训练步骤：随机 t -> 插值 x_t -> 预测速度 -> MSE。
    OT-CFM training step: random t -> interpolate x_t -> predict velocity -> MSE.
    Args:
        model: 速度场网络 velocity field network, forward(x_t, t) -> v
        x0: (B, D) noise samples
        x1: (B, D) data samples
    Returns:
        scalar MSE loss
    '''
    # TODO: implement
    # Hint:
    #   B = x0.shape[0]
    #   t = torch.rand(B, 1)
    #   x_t = ot_interpolate(x0, x1, t)
    #   v_target_val = velocity_target(x0, x1)
    #   v_pred = model(x_t, t)
    #   return F.mse_loss(v_pred, v_target_val)
    raise NotImplementedError('请实现 cfm_training_step')


# Uncomment after implementation:
# torch.manual_seed(0)
# net2 = VelocityNet()
# x0_ex = torch.randn(16, 2)
# x1_ex = torch.randn(16, 2)
# loss2 = cfm_training_step(net2, x0_ex, x1_ex)
# assert loss2.item() > 0
# print(f'\u2713 cfm_training_step loss={loss2.item():.4f}')
print('提示：实现 cfm_training_step 后取消注释运行验证')

## 延伸阅读 Further Reading

1. **Lipman et al. "Flow Matching for Generative Modeling" (ICLR 2023)**: <https://arxiv.org/abs/2210.02747> — Flow Matching 原始论文；第 3 节推导条件流匹配目标等价性。
2. **Tong et al. "Improving and Generalizing Flow-Matching through Optimal Transport" (TMLR 2024)**: <https://arxiv.org/abs/2302.00482> — OT-CFM；用最优传输耦合构造路径，减小方差，提升采样质量。
3. **Albergo & Vanden-Eijnden "Building Normalizing Flows with Stochastic Interpolants" (ICLR 2023)**: <https://arxiv.org/abs/2209.15571> — 随机插值（stochastic interpolants）框架，与 Flow Matching 并行提出。
4. **Song et al. "Score-Based Generative Modeling through Stochastic Differential Equations" (ICLR 2021)**: <https://arxiv.org/abs/2011.13456> — SDE/ODE 统一框架，概率流 ODE 的原始来源。
5. **Lilian Weng "What are Diffusion Models?" (2021)**: <https://lilianweng.github.io/posts/2021-07-11-diffusion-models/> — 系统性综述，含扩散 vs 流匹配对比。

## 关联练习 Related Assignments

```bash
python check.py w9-flow-matching
```

> 实现 `ot_interpolate(x0, x1, t)` 和 `velocity_target(x0, x1)`。
> 关键：`t` 可为 `(B, 1)` 张量或标量；`velocity_target` 返回与 $t$ 无关的常数向量场 `x1 - x0`。

> 能力标签：**GA3** · threshold ≥ 0.7